# Surface roughness: ESA WorldCover vs NLCD at Broadview

The Broadview wind field showed sharp localised dipoles with no terrain
signature. An A/B test tied them to isolated pixels ESA WorldCover labels
**built-up**: the wind field changed **33x more** within 150 m of one than
beyond 1.5 km.

The cause is a classification limit, not the model. WorldCover has a *single*
built-up class covering everything from a farm track to a city centre, and our
lookup maps it to an urban $z_0$ of 0.8 m. Against a grassland background at
0.03 m that is a **3.8σ jump across one 30 m pixel** — a spatial pattern
FuXi-CFD's training set (contiguous Chinese towns) essentially never contains.

NLCD splits development into four intensity classes, so this notebook asks one
question: **what does NLCD call the pixels WorldCover flags as built-up?**


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio

from fuxicfd_wrf.grids import Tile
from fuxicfd_wrf.terrain import nlcd
from fuxicfd_wrf.terrain.build import build_terrain
from fuxicfd_wrf.terrain.roughness import (
    CLASS_NAMES, NLCD_CLASS_NAMES, NLCD_Z0, WORLDCOVER_Z0,
)

TILE  = 'bv_center'
LONLAT = (-103.035, 34.817)
CACHE = Path('/scratch/moptis/fuxicfd/cache')

# NLCD is requester-pays and served as COGs, so we window-read one tile
# (a few hundred KB) rather than the 1.4 GB CONUS mosaic.
import os
os.environ.setdefault('AWS_REQUEST_PAYER', 'requester')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})


## 1. What does NLCD call WorldCover's 'built-up' pixels?


In [ ]:
import pyproj
tf = pyproj.Transformer.from_crs('EPSG:4326', 'EPSG:32613', always_xy=True)
x, y = tf.transform(*LONLAT)
tile = Tile(TILE, x=float(x), y=float(y), crs='EPSG:32613')

wc = np.rint(build_terrain(tile, CACHE, source='worldcover').landcover).astype(int)
nl = np.rint(build_terrain(tile, CACHE, source='nlcd').landcover).astype(int)

bu = wc == 50
print(f"WorldCover calls {bu.sum()} of {wc.size:,} pixels built-up "
      f"({100*bu.mean():.2f}%), all mapped to z0 = {WORLDCOVER_Z0[50]} m\n")
print(f"{'NLCD class':<32}{'px':>6}{'share':>8}{'NLCD z0':>9}")
for c, n in sorted(zip(*np.unique(nl[bu], return_counts=True)), key=lambda p: -p[1]):
    c = int(c)
    print(f'  {NLCD_CLASS_NAMES.get(c, c):<30}{n:6d}{100*n/bu.sum():7.1f}%{NLCD_Z0.get(c, float("nan")):9.3f}')


Three quarters land in **developed, open space** — NLCD's class for verges,
lawns and roadsides, typically <20% impervious, conventional $z_0\approx0.05$ m.
Not 0.8. A further slice is not developed at all.

## 2. Maps: the speckle in context

Two scales, because the problem needs both. The **context** view shows the two
datasets broadly agree on the landscape — this is not wholesale disagreement.
The **zoom** centres on the densest built-up cluster and shows what actually
differs.

Both $z_0$ panels share one scale and one ramp. The scale is **discrete**: $z_0$
comes from a class lookup and takes a handful of values, so a continuous log ramp
would spend its range on empty gaps and render 0.03 and 0.05 — the bulk of the
field — indistinguishable.


In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
from plot_roughness_sources import main as plot_sources

out = Path('/scratch/moptis/fuxicfd/out/roughness_sources.png')
plot_sources.__globals__  # noqa: B018 - keep the import obviously used
!python ../scripts/plot_roughness_sources.py --out {out}

from IPython.display import Image, display
display(Image(filename=str(out)))


The vertical dark line in the WorldCover zoom panel is a **road**. NLCD does not
treat it as urban fabric, and the difference panel shows deep blue along exactly
that line — NLCD roughly **16x lower** there (0.8 → 0.05 m).

Broad pale red elsewhere is NLCD reading rather more roughness than WorldCover
across the farmland (shrub/scrub at 0.10 where WorldCover sees grassland at
0.03). That is a real difference too, and unlike the road it is **not**
independently validated here — worth remembering before treating NLCD as truth
everywhere rather than better where it matters.

## 3. Effect on the wind field

From `scripts/ab_terrain.py --sources worldcover nlcd`, same 48 timesteps
through the model with only the roughness channel changed:

| distance to nearest WorldCover built-up px | mean \|Δ wind\| |
|---|---|
| 0–60 m | **0.170 m/s** |
| 60–150 m | 0.111 |
| 150–300 m | 0.062 |
| >1500 m | **0.023** |

**5.2x larger near the old speckle than far from it**, peak 1.04 m/s. The field
std falls 0.109 → 0.107 while the tile mean moves 0.002 m/s: spurious structure
removed, bulk flow untouched.

A caveat worth carrying: the far field moved too (0.023 m/s), so this is a
broader perturbation than a targeted despeckle, which showed 33x. NLCD
reclassifies everywhere, not only at the speckle.

## 4. The open question

FuXi-CFD was **trained on WorldCover-derived roughness**. The model consumes
$z_0$ in metres rather than class codes, so a more accurate $z_0$ should be a
better input — but this does shift the roughness channel relative to training,
and the authors' exact WorldCover→$z_0$ table is unknown, so our mapping already
differs from theirs by an unquantified amount.

`landcover_source` therefore defaults to `worldcover`. Phase 0 should arbitrate
it on evidence, the same way `dem_mode` was.
